# Phase 3, experiment 4: neighbour feature aggregates

**Why this experiment:** `05_weak_steps.ipynb` showed that the illicit transactions the baseline
misses are connected to other illicit transactions. Their own features do not give them away, but
their neighbourhood might. Degree features only count neighbours; they say nothing about what the
neighbours look like.

**The one change in this experiment:** for each transaction, add the average of its neighbours'
feature values.

- **Predecessors and successors are kept separate.** A predecessor is a transaction whose output this
  one spends (an incoming edge). A successor is a transaction that spends this one's output (an
  outgoing edge). Money coming in and money going out can mean different things.
- **Only the 20 most important raw features are averaged.** Averaging all 165 would add 330 columns.
  The 20 are chosen by the baseline model's feature importance, measured on the training steps only.
- **No labels are used**, so there is no leakage. The neighbours' features are available for every
  train and test transaction, including unlabeled ones.
- **Neighbours outside train/test are skipped**, because they have no features. A transaction with no
  known neighbour in a direction gets a missing value (NaN) there, which LightGBM handles by itself.

**Decisions:** keep the three degree features from notebook 03; start with the mean of
the neighbours' raw features on a subset of the most important features.

**How it is measured.** Notebook 04 showed that one run on this validation split is noisy (similar
models differ by 0.01 to 0.02). So each feature set is trained with the same four model settings as in
notebook 04, to see whether a change is consistent. Same split: train on steps 1 to 21, validate on 22 to 35.

## 1. Load

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

raw_cols = [c for c in train.columns if c.startswith("feat_")]
print("train:", train.shape, " test:", test.shape, " edges:", edges.shape)

train: (141772, 168)  test: (15329, 168)  edges: (234355, 2)


## 2. Degree features

In [ ]:
out_deg = edges["txId1"].value_counts()
in_deg = edges["txId2"].value_counts()

def add_degree_features(df):
    df = df.copy()
    df["in_degree"] = df["txId"].map(in_deg).fillna(0).astype(int)
    df["out_degree"] = df["txId"].map(out_deg).fillna(0).astype(int)
    df["total_degree"] = df["in_degree"] + df["out_degree"]
    return df

train = add_degree_features(train)
test = add_degree_features(test)
degree_cols = ["in_degree", "out_degree", "total_degree"]

## 3. Choose which features to aggregate

The baseline model is trained on the training steps (1 to 21) and its 20 most-used features are taken.
Only the training steps are used here so that the validation rows do not influence the choice.

In [ ]:
TRAIN_LAST_STEP = 21
N_TOP = 20   

is_labeled = train["label"].notna()
is_train_step = train["time_step"] <= TRAIN_LAST_STEP

selector = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
selector.fit(train.loc[is_labeled & is_train_step, raw_cols],
             train.loc[is_labeled & is_train_step, "label"].astype(int))

importance = pd.Series(selector.feature_importances_, index=raw_cols).sort_values(ascending=False)
top_cols = list(importance.head(N_TOP).index)
print(top_cols)

['feat_2', 'feat_53', 'feat_58', 'feat_138', 'feat_52', 'feat_16', 'feat_8', 'feat_142', 'feat_88', 'feat_5', 'feat_3', 'feat_89', 'feat_55', 'feat_100', 'feat_90', 'feat_125', 'feat_139', 'feat_23', 'feat_22', 'feat_161']


## 4. Neighbour aggregates

How the predecessor average is built, step by step:

1. Put the chosen features of every train and test transaction in one table, `tx_feats`.
2. Join it to the edge list on `txId1` (the source). Each edge row now carries the features of the
   source transaction. Edges whose source is outside train/test drop out of this join.
3. Group the rows by `txId2` (the destination) and take the mean. The result has one row per
   transaction, holding the average features of the transactions that feed into it.

The successor average is the same with the two columns swapped.

In [ ]:
# Step 1: chosen features for every transaction we have features for (train and test together).
# Edges never cross time steps, so train and test neighbourhoods do not mix.
tx_feats = pd.concat([train[["txId"] + top_cols], test[["txId"] + top_cols]])

# Predecessors: for edge (txId1 -> txId2), txId1 is a predecessor of txId2
# Step 2: attach the source transaction's features to each edge
pred_edges = edges.merge(tx_feats, left_on="txId1", right_on="txId")
# Step 3: average over all predecessors of each destination transaction
pred_mean = pred_edges.groupby("txId2")[top_cols].mean()
pred_mean.columns = ["pred_mean_" + c for c in top_cols]   # e.g. pred_mean_feat_2

# Successors: for edge (txId1 -> txId2), txId2 is a successor of txId1
succ_edges = edges.merge(tx_feats, left_on="txId2", right_on="txId")
succ_mean = succ_edges.groupby("txId1")[top_cols].mean()
succ_mean.columns = ["succ_mean_" + c for c in top_cols]

neigh_cols = list(pred_mean.columns) + list(succ_mean.columns)
print("New neighbour columns:", len(neigh_cols))

def add_neighbour_features(df):
    df = df.merge(pred_mean, left_on="txId", right_index=True, how="left")
    df = df.merge(succ_mean, left_on="txId", right_index=True, how="left")
    return df

train = add_neighbour_features(train)
test = add_neighbour_features(test)

print("Train rows with a known predecessor:", round(train[neigh_cols[0]].notna().mean(), 3))
print("Train rows with a known successor:  ", round(train[neigh_cols[-1]].notna().mean(), 3))
assert len(test) == 15329, "the join must not change the number of test rows"

New neighbour columns: 40
Train rows with a known predecessor: 0.744
Train rows with a known successor:   0.822


## 5. Split and feature sets

In [ ]:
labeled = train[train["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

tr = labeled[labeled["time_step"] <= TRAIN_LAST_STEP]
va = labeled[labeled["time_step"] > TRAIN_LAST_STEP]
y_tr, y_va = tr["label"], va["label"]

WEAK_STEPS = [24, 29, 31, 32]
OTHER_STEPS = [s for s in sorted(va["time_step"].unique()) if s not in WEAK_STEPS]

# Compare feature sets. Each one adds to the previous one.
feature_sets = {
    "raw": raw_cols,
    "raw+degree": raw_cols + degree_cols,
    "raw+degree+neigh": raw_cols + degree_cols + neigh_cols,
}
for name, cols in feature_sets.items():
    print(name, ":", len(cols), "features")

raw : 165 features
raw+degree : 168 features
raw+degree+neigh : 208 features


## 6. Train every combination (4 model settings x 3 feature sets)

In [ ]:
# Same four settings as notebook 04. 
model_settings = {
    "default": {},
    "leaves_15": {"num_leaves": 15},
    "leaves_7": {"num_leaves": 7},
    "slow": {"learning_rate": 0.03, "n_estimators": 300},
}

summary_rows = []    
per_step_auc = {}    # (setting name, feature set name) -> AUC for each validation step

for setting_name, params in model_settings.items():
    for fs_name, cols in feature_sets.items():
        model = lgb.LGBMClassifier(random_state=SEED, verbose=-1, **params)
        model.fit(tr[cols], y_tr)
        va_pred = model.predict_proba(va[cols])[:, 1]

        step_auc = {}
        for step in sorted(va["time_step"].unique()):
            in_step = (va["time_step"] == step).values
            step_auc[step] = roc_auc_score(y_va[in_step], va_pred[in_step])
        step_auc = pd.Series(step_auc)
        per_step_auc[(setting_name, fs_name)] = step_auc

        summary_rows.append({
            "setting": setting_name,
            "features": fs_name,
            "val_auc": roc_auc_score(y_va, va_pred),
            "mean_auc_weak_steps": step_auc[WEAK_STEPS].mean(),
            "mean_auc_other_steps": step_auc[OTHER_STEPS].mean(),
        })
        print("done:", setting_name, "/", fs_name)

summary = pd.DataFrame(summary_rows)

done: default / raw
done: default / raw+degree
done: default / raw+degree+neigh
done: leaves_15 / raw
done: leaves_15 / raw+degree
done: leaves_15 / raw+degree+neigh
done: leaves_7 / raw
done: leaves_7 / raw+degree
done: leaves_7 / raw+degree+neigh
done: slow / raw
done: slow / raw+degree
done: slow / raw+degree+neigh


## 7. Results

### 7.1 Validation AUC for each setting and feature set

`gain_from_neigh` is the change from adding the neighbour features on top of raw + degree. If the
neighbour features really help, it should be positive for all four settings, and larger than the
0.01 to 0.02 noise seen in notebook 04.

In [7]:
settings_order = list(model_settings.keys())
sets_order = list(feature_sets.keys())

val_table = summary.pivot(index="setting", columns="features", values="val_auc").loc[settings_order, sets_order]
val_table["gain_from_neigh"] = val_table["raw+degree+neigh"] - val_table["raw+degree"]
val_table.round(4)

features,raw,raw+degree,raw+degree+neigh,gain_from_neigh
setting,,,,
default,0.9148,0.8772,0.8923,0.0151
leaves_15,0.8937,0.8980,0.8949,-0.0031
leaves_7,0.9058,0.9067,0.9082,0.0014
slow,0.8929,0.8922,0.8936,0.0014


### 7.2 Average AUC on the weak steps (24, 29, 31, 32) and on the other ten steps

In [8]:
weak_table = summary.pivot(index="setting", columns="features", values="mean_auc_weak_steps").loc[settings_order, sets_order]
other_table = summary.pivot(index="setting", columns="features", values="mean_auc_other_steps").loc[settings_order, sets_order]

print("Mean AUC over the weak steps:")
display(weak_table.round(4))
print("Mean AUC over the other steps:")
display(other_table.round(4))

Mean AUC over the weak steps:


features,raw,raw+degree,raw+degree+neigh
setting,,,
default,0.7933,0.7306,0.7527
leaves_15,0.7613,0.7644,0.7557
leaves_7,0.7805,0.7820,0.7844
slow,0.7532,0.7553,0.7527


Mean AUC over the other steps:


features,raw,raw+degree,raw+degree+neigh
setting,,,
default,0.9866,0.9846,0.9850
leaves_15,0.9852,0.9861,0.9876
leaves_7,0.9851,0.9855,0.9864
slow,0.9851,0.9842,0.9859


### 7.3 Per-step AUC for the default setting

In [9]:
default_steps = pd.DataFrame({fs: per_step_auc[("default", fs)] for fs in sets_order})
default_steps["gain_from_neigh"] = default_steps["raw+degree+neigh"] - default_steps["raw+degree"]
default_steps.round(4)

,raw,raw+degree,raw+degree+neigh,gain_from_neigh
22,0.9991,0.9993,0.9990,-0.0003
23,0.9591,0.9532,0.9538,0.0005
24,0.7866,0.7426,0.7411,-0.0015
25,0.9997,0.9998,0.9998,0.0000
26,0.9312,0.9202,0.9212,0.0010
27,0.9995,0.9995,0.9995,0.0000
28,1.0000,1.0000,0.9999,-0.0001
29,0.7562,0.6943,0.7196,0.0253
30,0.9886,0.9897,0.9911,0.0013
31,0.7892,0.7191,0.7537,0.0346


## 8. Retrain on all labeled rows and save a submission

This uses the default model settings with the full feature set (raw + degree + neighbour).


In [10]:
final_cols = feature_sets["raw+degree+neigh"]

final_model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
final_model.fit(labeled[final_cols], labeled["label"])

test_pred = final_model.predict_proba(test[final_cols])[:, 1]
submission = pd.DataFrame({"index": test["index"], "target": test_pred})

# Format checks before saving
assert len(submission) == 15329, "wrong number of rows"
assert list(submission.columns) == ["index", "target"], "wrong columns"
assert submission["target"].notna().all(), "NaN in predictions"
assert (submission["index"].values == sample_sub["index"].values).all(), "index does not match sample submission"

SUB_PATH = "../submissions/sub03_neighbour.csv"
submission.to_csv(SUB_PATH, index=False)
print("Saved", SUB_PATH)

Saved ../submissions/sub03_neighbour.csv


## Results

- **Does adding neighbour features improve validation, and is it consistent across settings?** No.
  Compared with raw + degree, the change is +0.0014 (`leaves_7`), +0.0014 (`slow`), -0.0031
  (`leaves_15`) and +0.0151 (`default`). Three of the four are far below the 0.01 to 0.02 noise level
  found in notebook 04. The +0.0151 for the default setting mostly recovers the drop that the degree
  features caused for that setting, and at 0.8923 it is still below the raw-only default (0.9148).
- **What happens on the weak steps?** They stay weak. The average AUC over steps 24, 29, 31 and 32 is
  between 0.75 and 0.78 with neighbour features, the same range as without them. For the default
  setting, steps 29, 31 and 32 each rise by 0.025 to 0.035 compared with raw + degree, but remain
  below their raw-only scores, and step 24 does not move.
- **What happens on the other steps?** Almost nothing. Their average AUC is 0.985 to 0.988 for every
  feature set and setting.
- **Conclusion:** averaging the neighbours' raw features (top 20 features, predecessors and successors
  separately) gives no measurable benefit on this validation split.
- **Public leaderboard AUC for `sub03_neighbour.csv`:** 0.94584, against 0.94770 for raw + degree
  (`sub02_degree.csv`) and 0.94660 for the baseline (`sub01_baseline.csv`). That is 0.0019 lower than
  raw + degree and 0.0008 lower than the baseline. All three submissions are within 0.002 of each
  other, which is within noise for a score on 30% of the test set. The leaderboard agrees with
  validation: the neighbour features give no measurable benefit.
